# Laboratory Activity 6 — CNN Architecture Implementation

**Topic:** Convolutional Neural Networks using PyTorch

> This activity converts the CNN architecture diagram provided in `E1 - CNN Implementation.ipynb` (Laboratory Activity 6) into an equivalent PyTorch implementation. The network is constructed layer-by-layer while calculating and verifying the dimensions of the feature maps throughout the architecture.

---
## Standard Imports

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

## Reproducibility

In [2]:
def set_seed(seed: int) -> None:
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

---
## CNN Architecture Analysis

The diagram specifies a CNN dataset (28×28 grayscale sketches). The architecture has **four convolutional layers**, **two max-pooling layers**, a **dropout** layer, and **three fully connected layers**. Key details read directly from the diagram:

**Explicitly given:**
- Input shape: (1, 28, 28)
- Conv1: 32 filters, 3×3 kernel, stride 1, padding 1
- MaxPool1: 2×2 kernel, stride 2, **padding 1**
- Conv2: 64 filters, 3×3 kernel, stride 1, padding 1
- Conv3: 128 filters, 3×3 kernel, stride 1, padding 1
- Conv4: 256 filters, 3×3 kernel, stride 1, padding 1
- MaxPool2: 2×2 kernel, stride 2, padding 0
- Dropout: p = 0.2
- FCN1: input = ?, output = 1000
- FCN2: input = 1000, output = 500
- FCN3: input = 500, output = ?
- Activations: ReLU after each Conv and FC (except the last), Softmax on FCN3

**To be derived:**
- Spatial dimensions after each Conv/Pool (the "?" values)
- Flatten dimension → FCN1 input
- FCN3 output → number of classes

**Assumed:**
- FCN3 output = **10** (the Quick, Draw! dataset in this course uses 10 sketch categories, consistent with the MNIST demonstration)

### Layer Specification Table

| # | Layer | Input Shape | Configuration | Output Shape |
| ---: | :--- | :--- | :--- | :--- |
| 0 | Input | — | 28×28 grayscale | (B, 1, 28, 28) |
| 1 | Conv1 | (B, 1, 28, 28) | 32 filters, 3×3, s=1, p=1 | (B, 32, 28, 28) |
| 2 | ReLU | (B, 32, 28, 28) | — | (B, 32, 28, 28) |
| 3 | MaxPool1 | (B, 32, 28, 28) | 2×2, s=2, p=1 | (B, 32, 15, 15) |
| 4 | Conv2 | (B, 32, 15, 15) | 64 filters, 3×3, s=1, p=1 | (B, 64, 15, 15) |
| 5 | ReLU | (B, 64, 15, 15) | — | (B, 64, 15, 15) |
| 6 | Conv3 | (B, 64, 15, 15) | 128 filters, 3×3, s=1, p=1 | (B, 128, 15, 15) |
| 7 | ReLU | (B, 128, 15, 15) | — | (B, 128, 15, 15) |
| 8 | Conv4 | (B, 128, 15, 15) | 256 filters, 3×3, s=1, p=1 | (B, 256, 15, 15) |
| 9 | ReLU | (B, 256, 15, 15) | — | (B, 256, 15, 15) |
| 10 | MaxPool2 | (B, 256, 15, 15) | 2×2, s=2, p=0 | (B, 256, 7, 7) |
| 11 | Dropout | (B, 256, 7, 7) | p = 0.2 | (B, 256, 7, 7) |
| 12 | Flatten | (B, 256, 7, 7) | — | (B, 12544) |
| 13 | FCN1 + ReLU | (B, 12544) | 12544 → 1000 | (B, 1000) |
| 14 | FCN2 + ReLU | (B, 1000) | 1000 → 500 | (B, 500) |
| 15 | FCN3 + Softmax | (B, 500) | 500 → 10 | (B, 10) |

---
## CNN Components

**Convolutional Layer** — applies learnable filters (kernels) that slide across the input to extract spatial features such as edges, textures, and patterns. This architecture progressively increases the number of filters (32 → 64 → 128 → 256), learning increasingly complex features.

**ReLU Activation** — introduces non-linearity by setting negative values to zero:

$$ReLU(x) = \max(0, x)$$

**Max Pooling** — reduces spatial dimensions by selecting the maximum value in each pooling window. Note that MaxPool1 uses **padding=1**, which increases the spatial size before pooling, resulting in 15×15 output instead of 14×14.

**Dropout** — randomly sets a fraction ($p = 0.2$) of neurons to zero during training, which helps prevent overfitting.

**Flatten** — reshapes the 3-D feature maps (channels × height × width) into a 1-D vector so they can be processed by fully connected layers.

**Fully Connected (Linear) Layer** — combines the extracted features. The final layer produces one score per class, and Softmax converts these into probabilities.

---
## Feature-Map Dimension Calculations

The output size after a convolution or pooling operation (with dilation $D = 1$) is:

$$H_{out} = \left\lfloor \frac{H_{in} + 2P - K}{S} + 1 \right\rfloor$$

where $H_{in}$ = input size, $K$ = kernel size, $P$ = padding, $S$ = stride.

### Conv1: 3×3 kernel, stride 1, padding 1

$$\left\lfloor \frac{28 + 2(1) - 3}{1} + 1 \right\rfloor = \left\lfloor \frac{27}{1} + 1 \right\rfloor = 28$$

Output: $(B, 32, 28, 28)$ — spatial size is preserved due to same-padding.

### MaxPool1: 2×2 kernel, stride 2, padding 1

$$\left\lfloor \frac{28 + 2(1) - 2}{2} + 1 \right\rfloor = \left\lfloor \frac{28}{2} + 1 \right\rfloor = \left\lfloor 14 + 1 \right\rfloor = 15$$

Output: $(B, 32, 15, 15)$ — note that the padding=1 on this pooling layer produces 15×15 instead of the typical 14×14.

### Conv2, Conv3, Conv4: 3×3 kernel, stride 1, padding 1

All three convolutions use same-padding (kernel=3, padding=1), so the spatial dimensions remain unchanged:

$$\left\lfloor \frac{15 + 2(1) - 3}{1} + 1 \right\rfloor = \left\lfloor \frac{14}{1} + 1 \right\rfloor = 15$$

After Conv2: $(B, 64, 15, 15)$, after Conv3: $(B, 128, 15, 15)$, after Conv4: $(B, 256, 15, 15)$

### MaxPool2: 2×2 kernel, stride 2, padding 0

$$\left\lfloor \frac{15 + 2(0) - 2}{2} + 1 \right\rfloor = \left\lfloor \frac{13}{2} + 1 \right\rfloor = \left\lfloor 6.5 + 1 \right\rfloor = 7$$

Output: $(B, 256, 7, 7)$

### Verification with helper function

In [3]:
def calc_out(size, kernel_size, stride=1, padding=0):
    """Calculate spatial output size after convolution or pooling."""
    return ((size + 2 * padding - kernel_size) // stride) + 1

after_conv1 = calc_out(28, 3, 1, 1)       # Conv1
after_pool1 = calc_out(after_conv1, 2, 2, 1)  # MaxPool1 (padding=1!)
after_conv2 = calc_out(after_pool1, 3, 1, 1)  # Conv2
after_conv3 = calc_out(after_conv2, 3, 1, 1)  # Conv3
after_conv4 = calc_out(after_conv3, 3, 1, 1)  # Conv4
after_pool2 = calc_out(after_conv4, 2, 2, 0)  # MaxPool2

print(f"After Conv1:  {after_conv1}x{after_conv1}  (channels: 32)")
print(f"After Pool1:  {after_pool1}x{after_pool1}  (channels: 32)")
print(f"After Conv2:  {after_conv2}x{after_conv2}  (channels: 64)")
print(f"After Conv3:  {after_conv3}x{after_conv3}  (channels: 128)")
print(f"After Conv4:  {after_conv4}x{after_conv4}  (channels: 256)")
print(f"After Pool2:  {after_pool2}x{after_pool2}  (channels: 256)")

After Conv1:  28x28  (channels: 32)
After Pool1:  15x15  (channels: 32)
After Conv2:  15x15  (channels: 64)
After Conv3:  15x15  (channels: 128)
After Conv4:  15x15  (channels: 256)
After Pool2:  7x7  (channels: 256)


---
## Flatten Dimension

After MaxPool2, the feature maps have shape $(B, 256, 7, 7)$. The flatten dimension is:

$$C \times H \times W = 256 \times 7 \times 7 = 12{,}544$$

This becomes the `in_features` of FCN1.

In [4]:
flatten_dim = 256 * after_pool2 * after_pool2
print(f"Flatten dimension: 256 x {after_pool2} x {after_pool2} = {flatten_dim}")

Flatten dimension: 256 x 7 x 7 = 12544


---
## Filling in the Diagram’s Unknown Values

The architecture diagram leaves several values as **"?"** for the student to calculate:

| Diagram Notation | Calculated Value | Reasoning |
| :--- | :--- | :--- |
| Conv2 shape: (32, 64, ?, ?) | **15 × 15** | Same-padding preserves spatial size after Pool1 |
| Conv3 shape: (64, 128, ?, ?) | **15 × 15** | Same-padding preserves spatial size |
| Conv4 shape: (128, 256, ?, ?) | **15 × 15** | Same-padding preserves spatial size |
| Flatten shape: (32, ?) | **(32, 12544)** | $256 \times 7 \times 7 = 12{,}544$ |
| FCN1 input=? | **12544** | Equals the flatten dimension |
| FCN3 output=? | **10** | 10 sketch categories (assumed) |

---
## Shape Flow

```
Input              (B, 1, 28, 28)
     ↓
Conv1 + ReLU       (B, 32, 28, 28)
     ↓
MaxPool1 (p=1)     (B, 32, 15, 15)
     ↓
Conv2 + ReLU       (B, 64, 15, 15)
     ↓
Conv3 + ReLU       (B, 128, 15, 15)
     ↓
Conv4 + ReLU       (B, 256, 15, 15)
     ↓
MaxPool2 (p=0)     (B, 256, 7, 7)
     ↓
Dropout (p=0.2)    (B, 256, 7, 7)
     ↓
Flatten            (B, 12544)
     ↓
FCN1 + ReLU        (B, 1000)
     ↓
FCN2 + ReLU        (B, 500)
     ↓
FCN3 + Softmax     (B, 10)
```

---
## CNN Implementation

In [5]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()

        # --- Convolutional Block 1 ---
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2, padding=1)

        # --- Convolutional Block 2 ---
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1)
        self.conv4 = nn.Conv2d(128, 256, kernel_size=3, stride=1, padding=1)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2, padding=0)

        # --- Regularization ---
        self.dropout = nn.Dropout(p=0.2)

        # --- Fully Connected Layers ---
        self.fc1 = nn.Linear(256 * 7 * 7, 1000)  # 12544 -> 1000
        self.fc2 = nn.Linear(1000, 500)           # 1000  -> 500
        self.fc3 = nn.Linear(500, 10)             # 500   -> 10

    def forward(self, x):
        # Block 1: Conv1 -> ReLU -> MaxPool1
        x = F.relu(self.conv1(x))
        x = self.pool1(x)

        # Block 2: Conv2 -> ReLU -> Conv3 -> ReLU -> Conv4 -> ReLU -> MaxPool2
        x = F.relu(self.conv2(x))
        x = F.relu(self.conv3(x))
        x = F.relu(self.conv4(x))
        x = self.pool2(x)

        # Dropout
        x = self.dropout(x)

        # Flatten (preserve batch dimension)
        x = torch.flatten(x, 1)

        # Fully connected layers
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = F.softmax(self.fc3(x), dim=1)

        return x

---
## Instantiate the Model

In [6]:
model = CNN()
print(model)

CNN(
  (conv1): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool1): MaxPool2d(kernel_size=2, stride=2, padding=1, dilation=1, ceil_mode=False)
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv3): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv4): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (dropout): Dropout(p=0.2, inplace=False)
  (fc1): Linear(in_features=12544, out_features=1000, bias=True)
  (fc2): Linear(in_features=1000, out_features=500, bias=True)
  (fc3): Linear(in_features=500, out_features=10, bias=True)
)


---
## Parameter Count

For a convolutional layer: $(K_h \times K_w \times C_{in}) \times C_{out} + C_{out}$

For a fully connected layer: $N_{in} \times N_{out} + N_{out}$

| Layer | Calculation | Parameters |
| :--- | :--- | ---: |
| Conv1 | $(3 \times 3 \times 1) \times 32 + 32$ | 320 |
| Conv2 | $(3 \times 3 \times 32) \times 64 + 64$ | 18,496 |
| Conv3 | $(3 \times 3 \times 64) \times 128 + 128$ | 73,856 |
| Conv4 | $(3 \times 3 \times 128) \times 256 + 256$ | 295,168 |
| FCN1 | $12544 \times 1000 + 1000$ | 12,545,000 |
| FCN2 | $1000 \times 500 + 500$ | 500,500 |
| FCN3 | $500 \times 10 + 10$ | 5,010 |
| **Total** | | **13,438,350** |

In [7]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Trainable parameters: {count_parameters(model):,}")

Trainable parameters: 13,438,350


---
## Verify with Dummy Input

We pass a dummy tensor with the expected input shape through the network to confirm it executes without errors.

In [8]:
# Quick, Draw! images: 1 channel, 28x28
dummy_input = torch.randn(1, 1, 28, 28)

# Use eval mode so dropout is disabled during verification
model.eval()
with torch.no_grad():
    output = model(dummy_input)

print(f"Input shape:  {dummy_input.shape}")
print(f"Output shape: {output.shape}")

Input shape:  torch.Size([1, 1, 28, 28])
Output shape: torch.Size([1, 10])


---
## Intermediate Shape Verification

We pass the dummy input through each layer individually to verify that every intermediate tensor shape matches the manual calculations.

In [9]:
model.eval()
x = dummy_input

print(f"Input:            {x.shape}")

x = F.relu(model.conv1(x))
print(f"After Conv1+ReLU: {x.shape}")

x = model.pool1(x)
print(f"After Pool1:      {x.shape}")

x = F.relu(model.conv2(x))
print(f"After Conv2+ReLU: {x.shape}")

x = F.relu(model.conv3(x))
print(f"After Conv3+ReLU: {x.shape}")

x = F.relu(model.conv4(x))
print(f"After Conv4+ReLU: {x.shape}")

x = model.pool2(x)
print(f"After Pool2:      {x.shape}")

x = model.dropout(x)
print(f"After Dropout:    {x.shape}")

x = torch.flatten(x, 1)
print(f"After Flatten:    {x.shape}")

x = F.relu(model.fc1(x))
print(f"After FCN1+ReLU:  {x.shape}")

x = F.relu(model.fc2(x))
print(f"After FCN2+ReLU:  {x.shape}")

x = F.softmax(model.fc3(x), dim=1)
print(f"After FCN3+Soft:  {x.shape}")

Input:            torch.Size([1, 1, 28, 28])
After Conv1+ReLU: torch.Size([1, 32, 28, 28])
After Pool1:      torch.Size([1, 32, 15, 15])
After Conv2+ReLU: torch.Size([1, 64, 15, 15])
After Conv3+ReLU: torch.Size([1, 128, 15, 15])
After Conv4+ReLU: torch.Size([1, 256, 15, 15])
After Pool2:      torch.Size([1, 256, 7, 7])
After Dropout:    torch.Size([1, 256, 7, 7])
After Flatten:    torch.Size([1, 12544])
After FCN1+ReLU:  torch.Size([1, 1000])
After FCN2+ReLU:  torch.Size([1, 500])
After FCN3+Soft:  torch.Size([1, 10])


---
## Final Architecture Summary

| Layer | PyTorch Implementation | Output Shape |
| :--- | :--- | :--- |
| Input | — | (B, 1, 28, 28) |
| Conv1 + ReLU | `nn.Conv2d(1, 32, 3, stride=1, padding=1)` | (B, 32, 28, 28) |
| MaxPool1 | `nn.MaxPool2d(2, stride=2, padding=1)` | (B, 32, 15, 15) |
| Conv2 + ReLU | `nn.Conv2d(32, 64, 3, stride=1, padding=1)` | (B, 64, 15, 15) |
| Conv3 + ReLU | `nn.Conv2d(64, 128, 3, stride=1, padding=1)` | (B, 128, 15, 15) |
| Conv4 + ReLU | `nn.Conv2d(128, 256, 3, stride=1, padding=1)` | (B, 256, 15, 15) |
| MaxPool2 | `nn.MaxPool2d(2, stride=2, padding=0)` | (B, 256, 7, 7) |
| Dropout | `nn.Dropout(p=0.2)` | (B, 256, 7, 7) |
| Flatten | `torch.flatten(x, 1)` | (B, 12544) |
| FCN1 + ReLU | `nn.Linear(12544, 1000)` | (B, 1000) |
| FCN2 + ReLU | `nn.Linear(1000, 500)` | (B, 500) |
| FCN3 + Softmax | `nn.Linear(500, 10)` | (B, 10) |

---
## Network Explanation

The network accepts a single-channel 28×28 grayscale image as input. The first convolutional block applies 32 filters with a 3×3 kernel and same-padding, preserving the 28×28 spatial dimensions. MaxPool1 then reduces the size, but because it uses padding=1, the output is 15×15 rather than the typical 14×14.

The second convolutional block stacks three successive convolutions (Conv2, Conv3, Conv4), each using 3×3 kernels with same-padding. These progressively increase the number of feature channels from 32 → 64 → 128 → 256, learning increasingly abstract features while maintaining the 15×15 spatial resolution. MaxPool2 (no padding) then halves the spatial dimensions to 7×7.

A dropout layer (p=0.2) is applied after the last pooling to reduce overfitting by randomly deactivating 20% of neurons during training. The 256×7×7 feature maps are then flattened into a 12,544-element vector.

Three fully connected layers progressively reduce the representation: 12,544 → 1,000 → 500 → 10. ReLU activations follow FCN1 and FCN2, while Softmax is applied to the output of FCN3 to produce a probability distribution over the 10 classes.

---
## Conclusion

In this activity, the CNN architecture shown in the provided diagram was translated into PyTorch using `nn.Module`. The network contains four convolutional layers (with ReLU activations), two max-pooling layers (notably with different padding values), a dropout layer for regularization, and three fully connected layers ending with Softmax.

The spatial dimension calculations were performed using the standard formula, revealing that MaxPool1’s padding=1 yields 15×15 feature maps (rather than the more common 14×14), and that the final feature maps before flattening are 256×7×7 = 12,544 elements. A dummy forward pass confirmed that the implemented architecture produces the correct output shape at every stage, and the total number of trainable parameters was verified as 13,438,350.